In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "USERNAME_REMOVED"
password = "PASSWORD_REMOVED"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Branding block with logo and unique identifier
logo_block = html.Div(
    style={'textAlign': 'center', 'marginBottom': '10px'},
    children=[
        # Logo
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '80px'}
        ),
        html.Br(),
        # Unique identifier (edit this with your own name)
        html.Span(
            "Dashboard created by Lora Byrd",
            style={'fontWeight': 'bold', 'fontSize': '16px'}
        )
    ]
)

# Place the HTML image tag in the line below into the app.layout code according to your design
# Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

# Interactive filter controls (radio buttons)
filter_controls = html.Div(
    style={'textAlign': 'center', 'marginTop': '10px'},
    children=[
        html.Label("Rescue Type Filter:", style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'marginRight': '15px'}
        )
    ]
)

app.layout = html.Div([
    logo_block,
    
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Dashboard for Lora (aacuser)'))),
    html.Hr(),
    html.Div(
        
# Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        children=[
            filter_controls
        ]
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
# Set up the features for your interactive data table to make it user-friendly for your client
# If you completed the Module Six Assignment, you can copy in the code you created here 
# Interactive features for the client
                         page_size=10,
                         sort_action='native',
                         filter_action='native',
                         row_selectable='single',
                         selected_rows=[0],  # select the first row by default
        
                         style_table={'height': '400px', 'overflowY': 'auto'},
                         style_cell={
                         'whiteSpace': 'normal',
                         'height': 'auto',
                         'textAlign': 'left',
                         'padding': '5px'
                         }
                    ),

    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Rescue profile rules used by the scoring algorithm
RESCUE_PROFILES = {
    "water": {
        "animal_type": "Dog",
        "breeds": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ],
        "sex": "Intact Female",
        "min_age": 26,
        "max_age": 156
    },
    "mountain": {
        "animal_type": "Dog",
        "breeds": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_age": 26,
        "max_age": 156
    },
    "disaster": {
        "animal_type": "Dog",
        "breeds": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_age": 20,
        "max_age": 300
    }
}

def calculate_rescue_score(animal, rescue_type):
    """
    Calculates a rescue suitability score for an animal record.
    The score is based on animal type, breed, age range, and sex.
    """

    if rescue_type not in RESCUE_PROFILES:
        return 0

    profile = RESCUE_PROFILES[rescue_type]
    score = 0

    # Animal type match
    if animal.get("animal_type") == profile["animal_type"]:
        score += 25

    # Breed match
    if animal.get("breed") in profile["breeds"]:
        score += 35

    # Age range match
    age = animal.get("age_upon_outcome_in_weeks")

    if age is not None:
        try:
            age = float(age)

            if profile["min_age"] <= age <= profile["max_age"]:
                score += 25

        except (ValueError, TypeError):
            pass

    # Sex match
    if animal.get("sex_upon_outcome") == profile["sex"]:
        score += 15

    return score

@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    """
    Updates the dashboard based on the selected rescue type.
    The enhanced version ranks animals using a weighted rescue suitability score.
    """

    # Reset returns all records without scoring
    if filter_type == 'reset':
        results = db.read({})
        dff = pd.DataFrame.from_records(results)

        if '_id' in dff.columns:
            dff.drop(columns=['_id'], inplace=True)

        return dff.to_dict('records')

    # Read all animals so each record can be evaluated and scored
    results = db.read({})
    scored_animals = []

    for animal in results:
        score = calculate_rescue_score(animal, filter_type)

        # Only include animals with at least one matching criterion
        if score > 0:
            animal["match_score"] = score
            scored_animals.append(animal)

    # Sort animals from strongest match to weakest match
    scored_animals = sorted(
        scored_animals,
        key=lambda animal: animal.get("match_score", 0),
        reverse=True
    )

    dff = pd.DataFrame.from_records(scored_animals)

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Use current view of the DataTable; fall back to full df
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return [html.Div("No data available to display.")]

    fig = px.pie(
        dff,
        names='breed',
        title='Rescue Dogs by Breed'
    )

    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    # Prevent error on initial load
    if selected_columns is None:
        return []

    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, selected_rows):

    # If there is no data, return an empty list (no map)
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # If no row is selected or selection is empty, default to first row
    if not selected_rows:
        row = 0
    else:
        row = selected_rows[0]
        # If the selected row index is out of range after filtering, reset to 0
        if row >= len(dff):
            row = 0

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                # Marker with tool tip and popup
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 


<class 'ModuleNotFoundError'>: No module named 'jupyter_dash'